# QuIBL batch wrapper: deep-branch example

This notebook retains one representative analysis block from the study. The other branch-specific blocks used the same workflow with different clade definitions, outgroups, and output directories.

It is provided for methodological transparency rather than as a turnkey reproducibility workflow. Set the `ANALYSIS_ROOT` and `QUIBL_REPO` environment variables before adapting it to another directory structure.


In [ ]:
#!/usr/bin/env python3
# quibl_batch_wrapper.py
# Batch wrapper to run QuIBL over all 3-clade combinations and multiple gene tree inputs,
# combine results into a single table, annotate with species-tree support, and plot.

import os
import sys
import ast
import glob
import re
import subprocess
from itertools import combinations
from collections import Counter
from typing import Dict, List, Tuple, Optional
import numpy as np
import pandas as pd
from ete3 import Tree
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# ======================
# CONFIGURATION SECTION
# ======================

# These paths refer to the original analysis workspace, not the GitHub repository.
# Set ANALYSIS_ROOT and QUIBL_REPO in the environment before adapting this example.
ANALYSIS_ROOT = Path(os.environ.get("ANALYSIS_ROOT", "/path/to/original_analysis_workspace"))
DATA_DIR = ANALYSIS_ROOT / "data"
SCRATCH_DIR = ANALYSIS_ROOT / "scratch"

# 1) Define clades (name -> list of taxa). Names will be the collapsed labels in reduced trees.
CLADES: Dict[str, List[str]] = {
'Capitellida':['CATE','URUN'],
'Terebellida':['ARMA','TELA'],
'Clitellata':['EIAN','HEEU','PRLE'],
'Siboglinidae':['LABA','SIFI'],
'Cirratuliformia':['CICI','FLCO'],
'Spiosabelliformia':['BIPO','PRSP','NECE','STBE'],
# 'Other_Sedentaria':['LERO','PAHE','DISU','DIGY','SPSP','TRSP','MYSE'],
'Errantia':['LECL','PLDU','PABE','PHTR'],
# 'Pleistoannelida':['BIPO','LECL','CATE','CICI'],
'Sipuncula':['PHES','SINU'],
'Amphinomida':['PAJE','HECA'],
'Chaetopteridae':['CHDE','CHVA'] ,
'Magelonidae':['MAJO','MAMI'],
'Owenidae':['GAOC','OWFU'],
# 'Annelida_Outgroup':['CHSH','MIYE','LILO','TUPO','LIAN','PHAU'],
}

# 2) Gene tree inputs: list of files (one Newick per line) or directories with per-locus tree files
GENE_TREE_INPUTS: List[str] = [
    str(DATA_DIR / "trees/L42_synt.genetrees.renamed.nx"),
    str(DATA_DIR / "trees/L42_synt_C60.genetrees.renamed.nx"),
    str(DATA_DIR / "trees/L38_synt.genetrees.renamed.nx"),
    str(DATA_DIR / "trees/L38_synt_C60.genetrees.renamed.nx"),
]

# 3) QuIBL repo and base working/output directories
QUIBL_REPO = str(Path(os.environ.get("QUIBL_REPO", "/path/to/QuIBL")))
WORKDIR_BASE = str(SCRATCH_DIR / "tree_building/ILS/BatchQuIBL_deepbr")

# 4) Required/outgroup/root options
REQUIRED_TAXA: List[str] = ["LILO"]  # can be empty
TREE_ROOT = "LILO"
KEEP_OUTGROUPS_FOR_QUIBL = True
REQUIRE_ALL_REQUIRED_TAXA = False
ROOT_BY_TREE_ROOT = True

# 5) Filtering options
CLADES_REQUIRE_ALL = False
MIN_PRESENT_PER_CLADE = 1
FILTER_POLYTOMIES = True
FILTER_ZERO_LENGTH_INTERNALS = True
ENFORCE_MONO_EACH_CLADE = False

# 6) QuIBL parameters
QUIBL_NUM_DISTRIBUTIONS = 2
QUIBL_LIKELIHOOD_THRESH = 0.05
QUIBL_NUM_STEPS = 10
QUIBL_GRAD_ASCENT_SCALAR = 0.5
QUIBL_MULTIPROC = True
QUIBL_MAX_CORES = min(20, os.cpu_count() or 20)
# DeltaBIC = BIC1Dist - BIC2Dist
DELTA_BIC_THRESHOLD = 2.0

# 7) Stop early if too few trees remain after filtering per triplet+dataset
MIN_TREES_FOR_QUIBL = 5

# 8) Combined results path
COMBINED_RESULTS_CSV = os.path.join(WORKDIR_BASE, "quibl_combined_results.csv")

# 9) Environment activation for QuIBL run
ENV_RUNNER = ""  # "mamba", "conda", or "" to disable
QUIBL_ENV_NAME = "QuIBL"
SHELL_INIT = ""
SHELL_ACTIVATE_PREFIX = "conda activate"

# 10) Species tree for species-support annotation and plotting
SPECIES_TREE_PATH = str(DATA_DIR / "trees/L42.mycladetopo.topo29.with_node_labels.tree")

# 11) Plot options
MAKE_PLOT = True
PLOT_OUTPUT_PATH = os.path.join(WORKDIR_BASE, "quibl_deltaBIC_mixprop2_speciesaware.png")
JITTER_SIGMA_X = 0.2
JITTER_SIGMA_Y = 0.005

# ======================
# Helper functions
# ======================

def load_trees(trees_input: str) -> List[Tuple[Tree, str]]:
    """
    Returns a list of (tree_obj, source_id) pairs.
    - If trees_input is a directory, parses tree files in it (.treefile, .nwk, .newick, .tre, .tree).
    - If trees_input is a file, parses one Newick tree per non-empty line.
    """
    trees = []
    if os.path.isdir(trees_input):
        candidates = []
        for ext in (".treefile", ".nwk", ".newick", ".tre", ".tree"):
            candidates.extend(glob.glob(os.path.join(trees_input, f"*{ext}")))
        if not candidates:
            raise RuntimeError(f"No tree files found in directory: {trees_input}")
        for path in sorted(candidates):
            with open(path) as f:
                content = f.read().strip()
            lines = [l.strip() for l in content.splitlines() if l.strip()]
            nwk_line = None
            for l in reversed(lines):
                if l.startswith("(") and l.endswith(";"):
                    nwk_line = l
                    break
            if nwk_line is None and lines:
                nwk_line = lines[-1]
            if not nwk_line:
                continue
            try:
                t = Tree(nwk_line, format=1)
            except Exception as e:
                raise RuntimeError(f"Failed to parse {path}: {e}")
            trees.append((t, os.path.basename(path)))
    elif os.path.isfile(trees_input):
        with open(trees_input) as f:
            for idx, line in enumerate(f, start=1):
                s = line.strip()
                if not s:
                    continue
                try:
                    t = Tree(s, format=1)
                except Exception as e:
                    raise RuntimeError(f"Failed to parse line {idx} in {trees_input}: {e}")
                trees.append((t, f"line_{idx}"))
    else:
        raise RuntimeError("TREES_INPUT does not exist: " + trees_input)
    return trees


def any_polytomy(t: Tree) -> bool:
    for n in t.traverse():
        if not n.is_leaf() and len(n.children) != 2:
            return True
    return False


def has_zero_length_internal_branch(t: Tree) -> bool:
    for n in t.traverse():
        if n.is_leaf():
            continue
        for c in n.children:
            if c.dist is None or c.dist == 0.0:
                return True
    return False


def present_list(names: List[str], leaves: set) -> List[str]:
    return [x for x in names if x in leaves]


def root_by_tree_root_fn(t: Tree, tree_root: str, leaves: set) -> bool:
    if not tree_root or tree_root not in leaves:
        return False
    try:
        t.set_outgroup(tree_root)
        return True
    except Exception:
        return False


def monophyletic_present(t: Tree, clade_taxa: List[str], leaves: set) -> Tuple[bool, List[str]]:
    present = present_list(clade_taxa, leaves)
    if CLADES_REQUIRE_ALL and len(present) != len(clade_taxa):
        return False, present
    if len(present) < max(1, MIN_PRESENT_PER_CLADE):
        return False, present
    res = t.check_monophyly(values=present, target_attr="name")
    return bool(res[0] == True), present


def collapse_clade_mrca_to_leaf(t: Tree, clade_members_present: List[str], leaf_label: str) -> bool:
    if len(clade_members_present) == 0:
        return False
    try:
        mrca = t.get_common_ancestor(clade_members_present)
    except Exception:
        return False
    for child in list(mrca.children):
        child.detach()
    mrca.name = leaf_label
    return True


def prune_to_labels_and_required(t: Tree, labels: List[str], required_present: List[str]) -> bool:
    keep = set(labels) | set(required_present)
    names = [leaf.name for leaf in t.get_leaves() if leaf.name in keep]
    if any(lab not in names for lab in labels):
        return False
    t.prune(names, preserve_branch_length=True)
    return True


def prune_to_labels_and_optional_root(t: Tree, labels: List[str], tree_root: str) -> bool:
    leaves_now = set(leaf.name for leaf in t.get_leaves())
    root_present = [tree_root] if (tree_root and tree_root in leaves_now) else []
    return prune_to_labels_and_required(t, labels, root_present)


def find_quibl_table(outdir: str) -> str:
    cands = glob.glob(os.path.join(outdir, "*.csv")) + glob.glob(os.path.join(outdir, "*.tsv"))
    return cands[0] if cands else ""


def load_quibl_table(path: str) -> pd.DataFrame:
    if not path or not os.path.exists(path):
        return pd.DataFrame()
    sep = "," if path.endswith(".csv") else "\t"
    return pd.read_csv(path, sep=sep)


def dataset_id_from_path(path: str) -> str:
    if os.path.isdir(path):
        return os.path.basename(os.path.normpath(path))
    else:
        return os.path.splitext(os.path.basename(path))[0]


def col_ci(df: pd.DataFrame, name: str) -> Optional[str]:
    for c in df.columns:
        if c.lower() == name.lower():
            return c
    return None


def normalize_quibl_df(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize basic columns and compute DeltaBIC."""
    if df.empty:
        return df.copy()
    df = df.copy()

    # Standardize mixprop2 if present under variant names
    mix2 = None
    for cand in ["mixprop2", "mix_prop2", "MixProp2"]:
        c = col_ci(df, cand)
        if c is not None:
            mix2 = c
            break
    if mix2 and mix2 != "mixprop2":
        df.rename(columns={mix2: "mixprop2"}, inplace=True)

    # Standardize 'which' (pair/topology field)
    which = None
    for cand in ["which", "which_pair", "topology", "pair"]:
        c = col_ci(df, cand)
        if c is not None:
            which = c
            break
    if which and which != "which":
        df.rename(columns={which: "which"}, inplace=True)

    # Compute DeltaBIC = BIC1Dist - BIC2Dist if needed
    if "DeltaBIC" in df.columns:
        df["DeltaBIC"] = pd.to_numeric(df["DeltaBIC"], errors="coerce")
    elif set(["BIC1Dist", "BIC2Dist"]).issubset(df.columns):
        df["DeltaBIC"] = pd.to_numeric(df["BIC1Dist"], errors="coerce") - pd.to_numeric(df["BIC2Dist"], errors="coerce")
    else:
        df["DeltaBIC"] = np.nan

    # Convenience boolean
    df["Introgression_support"] = df["DeltaBIC"] > DELTA_BIC_THRESHOLD
    return df


# ======================
# Species-tree helper for sister-pair logic
# ======================

class SpeciesTreeHelper:
    def __init__(self, newick_path: str):
        # Load rooted species tree
        try:
            self.t = Tree(newick_path, format=1)
        except Exception:
            self.t = Tree(newick_path)
        self.root = self.t.get_tree_root()
        self.tipset = set([leaf.name for leaf in self.t.iter_leaves()])

    def has_tip(self, tip: str) -> bool:
        return tip in self.tipset

    def mrca_depth(self, labels: List[str]) -> Optional[float]:
        if not labels:
            return None
        node = self.t.get_common_ancestor(labels)
        # Topological distance from MRCA to root
        return float(node.get_distance(self.root, topology_only=True))


def tips_for_clade_label(clade_label: str, stree: SpeciesTreeHelper, clades: Dict[str, List[str]]) -> set:
    if clade_label in clades:
        return set([t for t in clades[clade_label] if stree.has_tip(t)])
    if stree.has_tip(clade_label):
        return {clade_label}
    return set()


def sister_pair_for_triplet(A_label: str, B_label: str, C_label: str, stree: SpeciesTreeHelper,
                            clades: Dict[str, List[str]], tol: float = 1e-9) -> Optional[frozenset]:
    SA = tips_for_clade_label(A_label, stree, clades)
    SB = tips_for_clade_label(B_label, stree, clades)
    SC = tips_for_clade_label(C_label, stree, clades)
    if not SA or not SB or not SC:
        return None
    if (SA & SB) or (SA & SC) or (SB & SC):
        return None  # overlapping definitions => ambiguous
    pairs = {
        ("A", "B"): list(SA | SB),
        ("A", "C"): list(SA | SC),
        ("B", "C"): list(SB | SC),
    }
    depths = {}
    for key, tips_union in pairs.items():
        d = stree.mrca_depth(tips_union)
        if d is None:
            return None
        depths[key] = d
    max_key = max(depths, key=lambda k: depths[k])
    max_val = depths[max_key]
    ties = [k for k, v in depths.items() if abs(v - max_val) <= tol]
    if len(ties) != 1:
        return None
    if max_key == ("A", "B"):
        return frozenset([A_label, B_label])
    if max_key == ("A", "C"):
        return frozenset([A_label, C_label])
    return frozenset([B_label, C_label])


def map_name_to_triplet_clade(name: str, A_label: str, B_label: str, C_label: str,
                              stree: SpeciesTreeHelper, clades: Dict[str, List[str]]) -> Optional[str]:
    name = str(name)
    if name == A_label:
        return A_label
    if name == B_label:
        return B_label
    if name == C_label:
        return C_label
    SA = tips_for_clade_label(A_label, stree, clades)
    SB = tips_for_clade_label(B_label, stree, clades)
    SC = tips_for_clade_label(C_label, stree, clades)
    hits = []
    if name in SA:
        hits.append(A_label)
    if name in SB:
        hits.append(B_label)
    if name in SC:
        hits.append(C_label)
    if len(hits) == 1:
        return hits[0]
    return None


def parse_pair_to_clade_labels(which_val: str, A_label: str, B_label: str, C_label: str,
                               stree: SpeciesTreeHelper, clades: Dict[str, List[str]]) -> Optional[frozenset]:
    """Return frozenset of two clade labels among {A_label,B_label,C_label}, or None."""
    s = str(which_val).strip()
    # Numeric codes like 12|3, 12, etc.
    digits = [d for d in s if d in "123"]
    if len(digits) >= 2:
        i, j = int(digits[0]), int(digits[1])
        lbls = [A_label, B_label, C_label]
        if 1 <= i <= 3 and 1 <= j <= 3 and i != j:
            return frozenset([lbls[i-1], lbls[j-1]])
    # Name-based left side before '|' or just two names separated by comma/space/semicolon/hyphen
    left = s.split("|", 1)[0]
    parts = [p for p in re.split(r"[,;\s]+", left) if p]
    if len(parts) >= 2:
        c1 = map_name_to_triplet_clade(parts[0], A_label, B_label, C_label, stree, clades)
        c2 = map_name_to_triplet_clade(parts[1], A_label, B_label, C_label, stree, clades)
        if c1 and c2 and c1 != c2:
            return frozenset([c1, c2])
    parts = [p for p in re.split(r"[-]+", left) if p]
    if len(parts) >= 2:
        c1 = map_name_to_triplet_clade(parts[0], A_label, B_label, C_label, stree, clades)
        c2 = map_name_to_triplet_clade(parts[1], A_label, B_label, C_label, stree, clades)
        if c1 and c2 and c1 != c2:
            return frozenset([c1, c2])
    return None

def run_triplet_for_dataset(
    dataset_path: str,
    triplet: Tuple[str, str, str],
    clades: Dict[str, List[str]],
    workdir_base: str,
) -> pd.DataFrame:
    """
    For one dataset and one clade triplet, run the full pipeline and return a result DataFrame
    with extra context columns. If a QuIBL results file already exists for this dataset+triplet,
    reuse it (do not recompute) and just parse/annotate the results.
    """
    ds_id = dataset_id_from_path(dataset_path)
    A, B, C = triplet
    triplet_tag = f"{A}__{B}__{C}"
    run_dir = os.path.join(workdir_base, ds_id, triplet_tag)
    os.makedirs(run_dir, exist_ok=True)

    # Paths for (possibly cached) outputs
    quibl_outdir = os.path.join(run_dir, "quibl_results")
    os.makedirs(quibl_outdir, exist_ok=True)
    quibl_output_csv = os.path.join(quibl_outdir, "quibl_results.csv")
    filtered_trees_path = os.path.join(run_dir, "filtered_collapsed_trees.nwk")
    quibl_config_path = os.path.join(run_dir, "myinput.txt")

    # If results already exist and are non-empty, reuse them without recomputation
    if os.path.exists(quibl_output_csv) and os.path.getsize(quibl_output_csv) > 0:
        df_raw = load_quibl_table(quibl_output_csv)
        if not df_raw.empty:
            # Normalize and enrich successful results
            df = normalize_quibl_df(df_raw)
            df["dataset"] = ds_id
            df["triplet"] = triplet_tag
            df["A"] = A
            df["B"] = B
            df["C"] = C
            # trees_kept: if filtered trees exist, count lines; else NaN
            if os.path.exists(filtered_trees_path):
                try:
                    with open(filtered_trees_path) as fh:
                        n_kept = sum(1 for ln in fh if ln.strip())
                except Exception:
                    n_kept = np.nan
            else:
                n_kept = np.nan
            df["trees_kept"] = n_kept
            df["skipped_quibl"] = False  # not "skipped"; we reused cached results
            df["quibl_failed"] = False
            df["error"] = "cached"
            if "discard_reasons_top" not in df.columns:
                df["discard_reasons_top"] = ""
            return df

    # No cached results or empty file: proceed to compute
    # Load trees
    trees = load_trees(dataset_path)

    # Filter and collapse
    filtered_collapsed, reasons = filter_and_collapse_for_triplet(
        trees=trees,
        triplet=triplet,
        clades=clades,
        required_taxa=REQUIRED_TAXA,
        tree_root=TREE_ROOT,
        keep_outgroups=KEEP_OUTGROUPS_FOR_QUIBL,
        enforce_monophyly=ENFORCE_MONO_EACH_CLADE,
    )

    # If too few trees, skip QuIBL and return a metadata row
    if len(filtered_collapsed) < MIN_TREES_FOR_QUIBL:
        meta = pd.DataFrame([{
            "dataset": ds_id,
            "triplet": triplet_tag,
            "A": A, "B": B, "C": C,
            "trees_kept": len(filtered_collapsed),
            "skipped_quibl": True,
            "quibl_failed": False,
            "error": "",
            "discard_reasons_top": str(dict(Counter(reasons).most_common(5))),
        }])
        return meta

    # Write filtered trees
    write_filtered_trees(filtered_collapsed, filtered_trees_path)

    # Prepare QuIBL config (points OutputPath to quibl_output_csv)
    build_quibl_config(
        config_path=quibl_config_path,
        treefile=filtered_trees_path,
        output_csv=quibl_output_csv,
        keep_outgroups=KEEP_OUTGROUPS_FOR_QUIBL,
        tree_root=TREE_ROOT
    )

    # Run QuIBL
    rc, out, err = run_quibl(QUIBL_REPO, quibl_config_path)

    # Read results
    # Prefer the expected output file; fall back to any CSV/TSV in outdir if needed
    res_table_path = quibl_output_csv if os.path.exists(quibl_output_csv) else find_quibl_table(quibl_outdir)
    df_raw = load_quibl_table(res_table_path)
    have_results = (not df_raw.empty)

    if not have_results:
        meta = pd.DataFrame([{
            "dataset": ds_id,
            "triplet": triplet_tag,
            "A": A, "B": B, "C": C,
            "trees_kept": len(filtered_collapsed),
            "skipped_quibl": False,
            "quibl_failed": True,
            "error": f"rc={rc}; stderr_tail={err[-500:] if err else ''}",
            "discard_reasons_top": str(dict(Counter(reasons).most_common(5))),
        }])
        return meta

    # Normalize and enrich successful results
    df = normalize_quibl_df(df_raw)
    df["dataset"] = ds_id
    df["triplet"] = triplet_tag
    df["A"] = A
    df["B"] = B
    df["C"] = C
    df["trees_kept"] = len(filtered_collapsed)
    df["skipped_quibl"] = False
    df["quibl_failed"] = False
    df["error"] = f"rc={rc}" if rc else ""
    df["discard_reasons_top"] = str(dict(Counter(reasons).most_common(5)))
    return df


def parse_triplet_labels(triplet_str: str) -> List[str]:
    """Split a triplet string 'A__B__C' into a 3-list of clade labels."""
    if triplet_str is None or not str(triplet_str).strip():
        return []
    parts = [p for p in str(triplet_str).strip().split("__") if p]
    return parts
    
def add_species_support(df: pd.DataFrame, stree_path: str, clades: Dict[str, List[str]]) -> pd.DataFrame:
    """
    Add columns:
      - sister_pair: frozenset of species-tree sister pair among {A,B,C} for each triplet (or None)
      - implicated_pair: frozenset of the two clades implicated by QuIBL for that row
        Prefer parsing from (triplet, outgroup); fallback to 'which' if needed.
      - species_support: True if implicated_pair == sister_pair, False if not, NaN if ambiguous
      - species_tree_topology: string like 'A,B|C' for the topology that matches the species tree
    """
    if df.empty:
        return df.copy()
    df = df.copy()
    stree = SpeciesTreeHelper(stree_path)

    # Cache species sister-pair per triplet
    sister_cache: Dict[str, Optional[frozenset]] = {}
    topo_cache: Dict[str, Optional[str]] = {}

    def species_topology_str(A: str, B: str, C: str, sis: Optional[frozenset]) -> Optional[str]:
        if sis is None:
            return None
        if set([A, B]) == set(sis):
            return f"{A},{B}|{C}"
        if set([A, C]) == set(sis):
            return f"{A},{C}|{B}"
        if set([B, C]) == set(sis):
            return f"{B},{C}|{A}"
        return None

    for triplet, sub in df.groupby("triplet"):
        A_label, B_label, C_label = sub["A"].iloc[0], sub["B"].iloc[0], sub["C"].iloc[0]
        sis = sister_pair_for_triplet(A_label, B_label, C_label, stree, clades)
        sister_cache[triplet] = sis
        topo_cache[triplet] = species_topology_str(A_label, B_label, C_label, sis)

    implicated_pairs = []
    species_support_vals = []
    use_outgroup = ("outgroup" in df.columns)

    for i, row in df.iterrows():
        trip = row.get("triplet", "")
        sis = sister_cache.get(trip)
        pair_fs = None

        # Preferred: derive implicated pair from triplet + outgroup
        if use_outgroup:
            pair_fs = implicated_pair_from_triplet_outgroup(trip, row.get("outgroup"))

        # Fallback: try legacy 'which' parsing if no outgroup or parsing failed
        if pair_fs is None and "which" in df.columns:
            A_label, B_label, C_label = row.get("A"), row.get("B"), row.get("C")
            pair_fs = parse_pair_to_clade_labels(row.get("which", ""), A_label, B_label, C_label, stree, clades)

        implicated_pairs.append(pair_fs)

        if pair_fs is None or sis is None:
            species_support_vals.append(np.nan)
        else:
            species_support_vals.append(pair_fs == sis)

    df["sister_pair"] = df["triplet"].map(sister_cache)
    df["implicated_pair"] = implicated_pairs
    df["species_support"] = species_support_vals
    df["species_tree_topology"] = df["triplet"].map(topo_cache)
    return df

# ======================
# QuIBL runner (robust to non-zero exit)
# ======================

def run_quibl(quibl_repo: str, config_path: str) -> Tuple[int, str, str]:
    """
    Run QuIBL and return (returncode, stdout, stderr) without raising on non-zero exit.
    Prefers mamba/conda 'run' if configured; falls back to shell activation or current Python.
    """
    quibl_py = os.path.join(quibl_repo, "QuIBL.py")
    if not os.path.exists(quibl_py):
        raise FileNotFoundError(f"QuIBL.py not found at {quibl_py}")

    def _run(cmd, use_shell=False, cwd=None):
        proc = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, shell=use_shell)
        return proc.returncode, proc.stdout, proc.stderr

    if ENV_RUNNER in ("mamba", "conda") and QUIBL_ENV_NAME:
        cmd = [ENV_RUNNER, "run", "-n", QUIBL_ENV_NAME, "python", quibl_py, config_path]
        print("Running QuIBL via:", " ".join(cmd))
        return _run(cmd, use_shell=False, cwd=quibl_repo)

    if SHELL_INIT and QUIBL_ENV_NAME:
        shell_cmd = f"{SHELL_INIT} && {SHELL_ACTIVATE_PREFIX} {QUIBL_ENV_NAME} && python {quibl_py} {config_path}"
        print("Running QuIBL via shell activation")
        return _run(["bash", "-lc", shell_cmd], use_shell=False, cwd=quibl_repo)

    cmd = [sys.executable, quibl_py, config_path]
    print("Running QuIBL via current Python:", " ".join(cmd))
    return _run(cmd, use_shell=False, cwd=quibl_repo)


# ======================
# Main pipeline pieces
# ======================

def filter_and_collapse_for_triplet(
    trees: List[Tuple[Tree, str]],
    triplet: Tuple[str, str, str],
    clades: Dict[str, List[str]],
    required_taxa: List[str],
    tree_root: str,
    keep_outgroups: bool,
    enforce_monophyly: bool,
) -> Tuple[List[Tuple[Tree, str]], Counter]:
    """
    For the given triplet of clade names, filter and collapse trees accordingly.
    Returns (filtered_collapsed_trees, discard_reasons_counter).
    """
    A, B, C = triplet
    cladeA_taxa = clades[A]
    cladeB_taxa = clades[B]
    cladeC_taxa = clades[C]
    labels = [A, B, C]

    reasons = Counter()
    filtered_collapsed = []

    for t_orig, src in trees:
        t = Tree(t_orig.write(format=1), format=1)
        leaves = {lf.name for lf in t.get_leaves()}

        required_present = present_list(required_taxa, leaves)
        if keep_outgroups:
            if tree_root and tree_root not in leaves:
                reasons["tree_root_missing"] += 1
                continue
            if REQUIRE_ALL_REQUIRED_TAXA:
                if len(required_present) != len(required_taxa):
                    reasons["missing_required_taxa"] += 1
                    continue
            else:
                if len(required_taxa) > 0 and len(required_present) == 0:
                    reasons["no_required_taxa_present"] += 1
                    continue

        if ROOT_BY_TREE_ROOT:
            root_by_tree_root_fn(t, tree_root, leaves)
            leaves = {lf.name for lf in t.get_leaves()}

        if FILTER_POLYTOMIES and any_polytomy(t):
            reasons["polytomy"] += 1
            continue
        if FILTER_ZERO_LENGTH_INTERNALS and has_zero_length_internal_branch(t):
            reasons["zero_length_internal"] += 1
            continue

        monoA, presA = monophyletic_present(t, cladeA_taxa, leaves)
        monoB, presB = monophyletic_present(t, cladeB_taxa, leaves)
        monoC, presC = monophyletic_present(t, cladeC_taxa, leaves)

        if enforce_monophyly:
            if not monoA:
                reasons["cladeA_not_monophyletic_or_insufficient"] += 1
                continue
            if not monoB:
                reasons["cladeB_not_monophyletic_or_insufficient"] += 1
                continue
            if not monoC:
                reasons["cladeC_not_monophyletic_or_insufficient"] += 1
                continue

        if len(presA) < 1:
            reasons["cladeA_no_members"] += 1
            continue
        if len(presB) < 1:
            reasons["cladeB_no_members"] += 1
            continue
        if len(presC) < 1:
            reasons["cladeC_no_members"] += 1
            continue

        okA = collapse_clade_mrca_to_leaf(t, presA, A)
        okB = collapse_clade_mrca_to_leaf(t, presB, B)
        okC = collapse_clade_mrca_to_leaf(t, presC, C)
        if not (okA and okB and okC):
            reasons["collapse_failed"] += 1
            continue

        if keep_outgroups:
            if not prune_to_labels_and_optional_root(t, labels, tree_root):
                reasons["prune_failed_or_missing_clade_labels"] += 1
                continue
        else:
            try:
                if any(lab not in {lf.name for lf in t.get_leaves()} for lab in labels):
                    reasons["missing_collapsed_labels_before_prune"] += 1
                    continue
                t.prune(labels, preserve_branch_length=True)
            except Exception:
                reasons["prune_failed_or_missing_clade_labels"] += 1
                continue

        if FILTER_POLYTOMIES and any_polytomy(t):
            reasons["polytomy_after_collapse"] += 1
            continue

        if keep_outgroups and tree_root:
            if tree_root not in {lf.name for lf in t.get_leaves()}:
                reasons["root_missing_after_prune"] += 1
                continue

        filtered_collapsed.append((t, src))

    return filtered_collapsed, reasons


def write_filtered_trees(filtered: List[Tuple[Tree, str]], out_path: str) -> None:
    with open(out_path, "w") as out:
        for t, _ in filtered:
            out.write(t.write(format=1) + "\n")


def build_quibl_config(
    config_path: str,
    treefile: str,
    output_csv: str,
    keep_outgroups: bool,
    tree_root: str,
) -> None:
    with open(config_path, "w") as f:
        f.write("[Input]\n")
        f.write(f"treefile: {os.path.abspath(treefile)}\n")
        f.write(f"numdistributions: {QUIBL_NUM_DISTRIBUTIONS}\n")
        f.write(f"likelihoodthresh: {QUIBL_LIKELIHOOD_THRESH}\n")
        f.write(f"numsteps: {QUIBL_NUM_STEPS}\n")
        f.write(f"gradascentscalar: {QUIBL_GRAD_ASCENT_SCALAR}\n")
        f.write(f"totaloutgroup: {tree_root if keep_outgroups else 0}\n")
        f.write(f"multiproc: {str(QUIBL_MULTIPROC)}\n")
        f.write(f"maxcores: {QUIBL_MAX_CORES}\n\n")
        f.write("[Output]\n")
        f.write(f"OutputPath: {os.path.abspath(output_csv)}\n")


def implicated_pair_from_triplet_outgroup(triplet_str: str, outgroup_str: str) -> Optional[frozenset]:
    """
    From triplet 'A__B__C' and outgroup 'A|B|C', return frozenset of the two non-outgroup clades.
    Returns None if parsing fails or outgroup not in the triplet.
    """
    labs = parse_triplet_labels(triplet_str)
    if len(labs) != 3 or outgroup_str is None:
        return None
    outg = str(outgroup_str).strip()
    if outg not in labs:
        return None
    pair = [x for x in labs if x != outg]
    if len(pair) != 2:
        return None
    return frozenset(pair)
# Robust pair label builder that accepts Python containers or their string representations
def pair_to_label(val, row=None):
    # If we already have a container with two items
    if isinstance(val, (set, frozenset, list, tuple)):
        items = [str(x) for x in val]
        if len(items) == 2:
            a, b = sorted(items)
            return f"{a} <-> {b}"
    # If it's a string, try to parse
    s = str(val).strip()
    if s:
        # Handle "frozenset({...})" by extracting the inner literal
        m = re.match(r"^\s*frozenset\((.*)\)\s*$", s)
        inner = m.group(1) if m else s
        try:
            obj = ast.literal_eval(inner)
            if isinstance(obj, (set, frozenset, list, tuple)):
                items = [str(x) for x in obj]
                if len(items) == 2:
                    a, b = sorted(items)
                    return f"{a} <-> {b}"
        except Exception:
            pass
    # Last fallback: recompute from triplet+outgroup if available
    if row is not None and ("triplet" in row) and ("outgroup" in row):
        fs = implicated_pair_from_triplet_outgroup(row["triplet"], row["outgroup"])
        if isinstance(fs, (set, frozenset)) and len(fs) == 2:
            a, b = sorted(list(fs))
            return f"{a} <-> {b}"
    return None

def plot_scatter(all_df: pd.DataFrame, out_path: str, var='mixprop2') -> None:
    plot_df = all_df.copy()

    # Human-readable support labels for hue
    def support_label(val):
        if pd.isna(val):
            return "Ambiguous"
        return "Species-supporting" if val else "Non-species"
    plot_df["support_label"] = plot_df["species_support"].apply(support_label)

    # Build label text from implicated_pair, with robust parsing
    plot_df["label_text"] = plot_df.apply(lambda r: pair_to_label(r.get("implicated_pair"), r), axis=1)

    # Jitter to reduce overplotting; reuse these for annotations
    # Use configured JITTER_SIGMA_*, else defaults
    _jit_x = JITTER_SIGMA_X if "JITTER_SIGMA_X" in globals() else 0.2
    _jit_y = JITTER_SIGMA_Y if "JITTER_SIGMA_Y" in globals() else 0.005
    rng = np.random.default_rng(42)
    plot_df["DeltaBIC_jit"] = plot_df["DeltaBIC"] + rng.normal(0, _jit_x, size=len(plot_df))
    plot_df["mixprop2_jit"] = plot_df[var] + rng.normal(0, _jit_y, size=len(plot_df))

    # Scatter
    sns.set(style="whitegrid")
    plt.figure(figsize=(10, 6))
    ax = sns.scatterplot(
        data=plot_df,
        x="DeltaBIC_jit", y="mixprop2_jit",
        hue="support_label",
        style="dataset",
        palette={"Species-supporting": "#1f77b4", "Non-species": "#d62728", "Ambiguous": "#bbbbbb"},
        alpha=0.85, s=60
    )
    # Reference lines
    for x in [10]:
        plt.axvline(x=x, color="k", linestyle="--", linewidth=1, alpha=0.6)

    ax.set_xlabel("DeltaBIC (BIC1Dist − BIC2Dist)")
    ax.set_ylabel(var)
    ax.set_title(f"QuIBL introgression summary: DeltaBIC vs {var} (species-tree aware)")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)

    # Annotate strongest non-species cases: DeltaBIC > 10 and species_support == False
    anno_df = plot_df[
        (plot_df["DeltaBIC"] > 10) &
        (plot_df["species_support"].eq(False)) &
        (plot_df["label_text"].notna())
    ].copy()

    # Small positional nudge for labels to reduce overlap
    nudge_x = 0.05
    nudge_y = 0.01
    if not anno_df.empty:
        for _, r in anno_df.iterrows():
            ax.text(
                r["DeltaBIC_jit"] + nudge_x,
                r["mixprop2_jit"] + nudge_y,
                r["label_text"],
                fontsize=8,
                color="#333333",
                ha="left",
                va="bottom"
            )

    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.savefig(os.path.splitext(out_path)[0] + ".svg")

    plt.show()


def add_mixprop2_parts(combined_df: pd.DataFrame) -> pd.DataFrame:
    """
    Add two columns to combined_df:
      - mixprop2_all: proportion of introgressed loci among all loci
                      = mixprop2 * (count / trees_kept)
      - mixprop2_discord: proportion of introgressed loci among discordant loci only
                          = mixprop2 * (count / sum(count over discordant rows in the same group))
        For the species-supporting row, this is set to "n.a." (string) for display.

    Grouping is by (dataset, triplet) if 'dataset' exists, else by 'triplet'.
    """
    required = {"mixprop2", "count", "trees_kept", "triplet", "species_support"}
    missing = [c for c in required if c not in combined_df.columns]
    if missing:
        raise ValueError(f"add_mixprop2_parts: missing required columns: {missing}")

    df = combined_df.copy()

    # Ensure numeric types
    for col in ["mixprop2", "count", "trees_kept"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # mixprop2 among all loci
    df["mixprop2_all"] = df["mixprop2"] * (df["count"] / df["trees_kept"])
    # Clean up infinities from divide-by-zero
    df.loc[~np.isfinite(df["mixprop2_all"]), "mixprop2_all"] = np.nan

    # Determine grouping columns
    group_cols = ["dataset", "triplet"] if "dataset" in df.columns else ["triplet"]

    # Compute mixprop2 among discordant loci only
    def per_group(g: pd.DataFrame) -> pd.Series:
        discord_mask = g["species_support"].eq(False)
        discord_total = g.loc[discord_mask, "count"].sum(min_count=1)  # NaN if none
        out = pd.Series(np.nan, index=g.index, dtype="float64")
        if pd.notna(discord_total) and discord_total > 0:
            out.loc[discord_mask] = (
                g.loc[discord_mask, "mixprop2"] * (g.loc[discord_mask, "count"] / discord_total)
            )
        return out

    df["mixprop2_discord"] = df.groupby(group_cols, group_keys=False).apply(per_group)

    # Display-friendly string for species-supporting rows (matches example)
    df.loc[df["species_support"].eq(True), "mixprop2_discord"] = np.nan

    return df


def add_count_summaries(combined_df: pd.DataFrame) -> pd.DataFrame:
    """
    Add two columns to combined_df:
      - count_all: total number of trees kept for the triplet (trees_kept), broadcast to all rows
      - count_discord: sum of 'count' over the two discordant rows for that triplet (and dataset, if present),
                       broadcast to all rows in the group. NaN if no discordant rows.

    Grouping is by (dataset, triplet) if 'dataset' exists; otherwise by 'triplet'.
    """
    required = {"count", "trees_kept", "triplet", "species_support"}
    missing = [c for c in required if c not in combined_df.columns]
    if missing:
        raise ValueError(f"add_count_summaries: missing required columns: {missing}")

    df = combined_df.copy()

    # Ensure numeric types where appropriate
    df["count"] = pd.to_numeric(df["count"], errors="coerce")
    df["trees_kept"] = pd.to_numeric(df["trees_kept"], errors="coerce")

    # count_all is simply trees_kept broadcast to all rows
    df["count_all"] = df["trees_kept"]

    # Determine grouping columns
    group_cols = ["dataset", "triplet"] if "dataset" in df.columns else ["triplet"]

    # Build a boolean mask for discordant rows (species_support == False), robust to strings/booleans
    species_support_str = df["species_support"].astype(str).str.lower()
    discord_mask = (species_support_str == "false") | (df["species_support"] == False)

    # Sum 'count' across discordant rows within each group and broadcast
    discord_counts_only = df["count"].where(discord_mask)
    if len(group_cols) == 1:
        df["count_discord"] = discord_counts_only.groupby(df[group_cols[0]]).transform("sum")
    else:
        df["count_discord"] = discord_counts_only.groupby([df[c] for c in group_cols]).transform("sum")

    # Keep numeric dtype; NaN signifies no discordant rows in the group
    df["count_discord"] = pd.to_numeric(df["count_discord"], errors="coerce")

    return df

def main():
    os.makedirs(WORKDIR_BASE, exist_ok=True)

    all_triplets = list(combinations(sorted(CLADES.keys()), 3))
    combined: List[pd.DataFrame] = []

    for ds in GENE_TREE_INPUTS:
        for trip in all_triplets:
            print(f"Processing dataset={dataset_id_from_path(ds)} triplet={trip} ...")
            df_run = run_triplet_for_dataset(ds, trip, CLADES, WORKDIR_BASE)
            if df_run is None or df_run.empty:
                df_run = pd.DataFrame([{
                    "dataset": dataset_id_from_path(ds),
                    "triplet": "__".join(trip),
                    "A": trip[0], "B": trip[1], "C": trip[2],
                    "trees_kept": 0,
                    "skipped_quibl": True,
                    "quibl_failed": False,
                    "error": "",
                    "discard_reasons_top": "{}"
                }])
            combined.append(df_run)

    combined_df = pd.concat(combined, ignore_index=True, sort=False)

    # Normalize presence/columns for plotting
    if "Introgression_support" in combined_df.columns:
        combined_df["Introgression_support"] = combined_df["Introgression_support"].astype(bool)
    # Ensure expected columns exist for plotting, even if all NaN
    if "mixprop2" not in combined_df.columns:
        combined_df["mixprop2"] = np.nan
    if "which" not in combined_df.columns:
        combined_df["which"] = ""

    # Species-tree aware annotation
    combined_df = add_species_support(combined_df, SPECIES_TREE_PATH, CLADES)
    combined_df = combined_df.drop_duplicates()
    combined_df = add_mixprop2_parts(combined_df)
    combined_df = add_count_summaries(combined_df)

    combined_df.to_csv(COMBINED_RESULTS_CSV, index=False)
    print(f"Combined results written to: {COMBINED_RESULTS_CSV}")
    print(f"Total rows: {len(combined_df)}")

    # Plot
    if MAKE_PLOT:
        combined_df = pd.read_csv(COMBINED_RESULTS_CSV)
        # Keep rows with both DeltaBIC and mixprop2 available
        plotable = combined_df.dropna(subset=["DeltaBIC", "mixprop2"])
        if plotable.empty:
            print("No rows with both DeltaBIC and mixprop2 to plot.")
        else:
            plot_scatter(plotable, PLOT_OUTPUT_PATH, var="mixprop2_all")


if __name__ == "__main__":
    main()